# Module 6 Loops — Advanced Level


This notebook covers five advanced loop techniques that you will use regularly in security scripting: `enumerate()`, `zip()`, the `for…else` clause, advanced `while` patterns, and nested loop control.

## Table of Contents

- [ ] 1. `enumerate()` — Index and Value Together
- [ ] 2. `zip()` — Looping in Parallel
- [ ] 3. The `for…else` Clause
- [ ] 4. Advanced `while` Patterns
- [ ] 5. Nested Loop Control
- [ ] Summary

---
## 1. `enumerate()` — Index and Value Together

### The problem: tracking position manually

Sometimes you need to know *where* you are in a loop — not just the current value. Without `enumerate()`, you would manage a counter manually:

```python
index = 0
for port_number in range(80, 85):
    print(f"Port #{index}: {port_number}")
    index += 1   # easy to forget, easy to get wrong
```

### `enumerate()` — the clean solution

`enumerate()` wraps a sequence and yields a **(index, value)** pair on each iteration:

```python
for index, value in enumerate(sequence):
    ...
```

By default the index starts at 0. You can change that with `start=`:

```python
for index, value in enumerate(sequence, start=1):
    ...
```

### Security use case

Numbering findings in a scan report, tracking which attempt number produced a result, generating numbered log lines.

In [ ]:
# enumerate() with range(): number each port in a scan result
# index gives the finding number, port_number gives the actual port

for finding_number, port_number in enumerate(range(80, 86), start=1):
    print(f"Finding #{finding_number}: port {port_number}")

In [ ]:
# enumerate() with a string: number each character in a CVE ID
# Useful for building position-aware parsers

cve_id = "CVE-2024-1234"

for position, character in enumerate(cve_id):
    print(f"Position {position}: '{character}'")

### ✅ Check Your Understanding: `enumerate()`

#### Exercise 1 — Predict

Before running, predict the output:

```python
for idx, score in enumerate(range(7, 11), start=1):
    print(f"Finding {idx}: CVSS {score}")
```

How many lines? What does line 2 say?

In [ ]:
# Exercise 1 — Predict the output, then run to verify
# My prediction:
#

for idx, score in enumerate(range(7, 11), start=1):
    print(f"Finding {idx}: CVSS {score}")

<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
# range(7, 11) produces 7, 8, 9, 10 — 4 values
# enumerate starts idx at 1
# Output:
#   Finding 1: CVSS 7
#   Finding 2: CVSS 8
#   Finding 3: CVSS 9
#   Finding 4: CVSS 10
```

</details>

#### Exercise 2 — Write

Use `enumerate()` to loop through the string `"HTTPS"`, starting the index at 1.

For each character print: `"Character {index}: {char}"`

In [ ]:
# Exercise 2 — Enumerate characters in a protocol name
# Your code here:


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
# 'HTTPS' has 5 characters; enumerate starts at 1

for char_index, character in enumerate("HTTPS", start=1):
    print(f"Character {char_index}: {character}")
```

</details>

---
## 2. `zip()` — Looping in Parallel

### What is `zip()`?

`zip()` combines two or more sequences and iterates through all of them **simultaneously** — one element from each per iteration.

```python
for value_a, value_b in zip(sequence_a, sequence_b):
    ...
```

### The stopping rule

`zip()` stops as soon as the **shortest** sequence is exhausted. Extra elements in longer sequences are silently ignored.

### Security use case

Pairing a list of port numbers with a list of expected services, pairing scan results with severity levels, combining two related data sources.

In [ ]:
# zip(): pair port numbers with their service names
# Both sequences have the same length — zip pairs them element by element

port_numbers  = range(0, 5)           # 0, 1, 2, 3, 4
service_names = ("Reserved", "tcpmux", "compressnet", "compressnet", "unassigned")

for port_number, service_name in zip(port_numbers, service_names):
    print(f"Port {port_number}: {service_name}")

In [ ]:
# zip() with unequal lengths: the shorter sequence wins
# severity_levels has only 3 items — the 4th port_number is never reached

port_numbers    = range(80, 84)              # 80, 81, 82, 83
severity_levels = ("HIGH", "MEDIUM", "LOW")  # only 3 items

for port_number, severity in zip(port_numbers, severity_levels):
    print(f"Port {port_number} — severity: {severity}")

print("(Port 83 was never reached — severity_levels ran out first)")

### ✅ Check Your Understanding: `zip()`

#### Exercise 3 — Fix

The code below is meant to pair each CVSS score with its severity label. It runs without error but produces wrong output. Find and fix the bug.

```python
cvss_scores    = range(6, 10)                        # 6, 7, 8, 9
severity_labels = ("MEDIUM", "HIGH", "HIGH", "CRITICAL")

for severity, score in zip(cvss_scores, severity_labels):
    print(f"CVSS {score} — {severity}")
```

In [ ]:
# Exercise 3 — Fix the variable order in the zip unpacking
# Your corrected code here:


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
# Bug: the unpacking order is wrong.
# zip(cvss_scores, severity_labels) yields (score, label) pairs.
# The loop variable names were swapped — 'severity' was receiving the score and vice versa.

cvss_scores     = range(6, 10)
severity_labels = ("MEDIUM", "HIGH", "HIGH", "CRITICAL")

for score, severity in zip(cvss_scores, severity_labels):
    print(f"CVSS {score} — {severity}")
```

</details>

#### Exercise 4 — Write

Use `zip()` to pair each port number in `range(22, 26)` with its matching protocol from the tuple `("SSH", "SMTP", "SMTPS", "SMTP-alt")`.

Print: `"Port {n} — {protocol}"`

In [ ]:
# Exercise 4 — Pair port numbers with protocol names using zip()
# Your code here:


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
# range(22, 26) produces 22, 23, 24, 25 — 4 values matching the 4-element tuple

protocols = ("SSH", "SMTP", "SMTPS", "SMTP-alt")

for port_number, protocol in zip(range(22, 26), protocols):
    print(f"Port {port_number} — {protocol}")
```

</details>

---
## 3. The `for…else` Clause

### What is `for…else`?

Python allows an `else` block after a `for` (or `while`) loop. The `else` block runs **only if the loop completed normally** — meaning it was never interrupted by a `break`.

```python
for item in sequence:
    if some_condition:
        break
else:
    # only runs if the loop finished WITHOUT hitting break
    ...
```

### Why this is useful

The classic use case is a **search** loop:
- If you find what you are looking for — `break` out and handle the finding
- If you reach the end without finding it — the `else` block runs and handles the 'not found' case

This avoids needing a separate `found = False` flag variable.

In [ ]:
# for...else: scan for a forbidden port; report clearly whether it was found or not
# The else block only runs if no break occurred (i.e. port 21 was NOT found)

forbidden_port = 21  # FTP — unencrypted file transfer

for port_number in range(20, 25):
    print(f"Checking port {port_number}")
    if port_number == forbidden_port:
        print(f"ALERT: FTP (port {forbidden_port}) is open — immediate action required")
        break
else:
    # This line only runs if the loop finished without hitting break
    print("No forbidden ports detected in range")

In [ ]:
# Contrast: same structure but forbidden port is outside the range
# The break never fires, so the else block runs

forbidden_port = 21

for port_number in range(80, 85):
    print(f"Checking port {port_number}")
    if port_number == forbidden_port:
        print(f"ALERT: FTP detected")
        break
else:
    print("Clean scan: no forbidden ports detected")

### ✅ Check Your Understanding: `for…else`

#### Exercise 5 — Predict

Before running, predict what the code below prints. Will the `else` block execute?

```python
target_score = 9

for cvss_score in range(1, 8):
    if cvss_score == target_score:
        print(f"Critical score {target_score} found")
        break
else:
    print("No critical score in range")
```

In [ ]:
# Exercise 5 — Predict the output, then run to verify
# My prediction:
#

target_score = 9

for cvss_score in range(1, 8):
    if cvss_score == target_score:
        print(f"Critical score {target_score} found")
        break
else:
    print("No critical score in range")

<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
# range(1, 8) produces 1, 2, 3, 4, 5, 6, 7 — it never reaches 9.
# The condition (cvss_score == 9) is never True — break never fires.
# The loop completes normally, so the else block runs.
# Output: No critical score in range
```

</details>

#### Exercise 6 — Write

Write a `for…else` loop over `range(1025, 1030)` that:
- Checks if any port number equals `1027`
- If found: prints `"Port 1027 detected"` and breaks
- If not found (else): prints `"Port 1027 not in range"`

In [ ]:
# Exercise 6 — Use for...else to search for a specific port
# Your code here:


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
# range(1025, 1030) contains 1027, so the break fires and else is skipped

for port_number in range(1025, 1030):
    if port_number == 1027:
        print("Port 1027 detected")
        break
else:
    print("Port 1027 not in range")
```

</details>

---
## 4. Advanced `while` Patterns

### Pattern 1: Retry with a maximum limit

A common real-world need: keep trying something until it succeeds OR until you hit a maximum number of attempts. This uses a `while` loop with **two exit conditions**:
- Normal exit: the maximum is reached (counter hits the limit)
- Early exit: success happens (`break`)

```python
attempt = 0
while attempt < max_attempts:
    attempt += 1
    if success_condition:
        break
```

In [ ]:
# Pattern 1: retry connection to a port with a max attempt limit
# Simulates an SSH connection attempt that succeeds on attempt 3

max_attempts    = 5
success_attempt = 3  # simulated: connection succeeds on this attempt
attempt_number  = 0

while attempt_number < max_attempts:
    attempt_number += 1
    print(f"Attempt {attempt_number}: connecting to port 22...")

    if attempt_number == success_attempt:
        print(f"Connection established on attempt {attempt_number}")
        break
else:
    # while loops also support else — runs only if the loop ended WITHOUT break
    print("All attempts exhausted — host unreachable")

### Pattern 2: Process until a sentinel value

A **sentinel value** is a special value that signals 'stop'. Instead of counting, the loop runs until it encounters the sentinel.

Common in security tools that process items from a queue or stream until a 'done' signal arrives.

In [ ]:
# Pattern 2: process CVSS scores until the sentinel value 0 is reached
# 0 is the sentinel — it signals end of input, not a real score

cvss_scores_stream = (9, 7, 4, 8, 0, 6)  # 0 is the stop signal
processed_count = 0
score_index = 0

while score_index < len(cvss_scores_stream):
    current_score = cvss_scores_stream[score_index]

    if current_score == 0:  # sentinel reached
        break

    print(f"Processing CVSS score: {current_score}")
    processed_count += 1
    score_index += 1

print(f"Processed {processed_count} scores before sentinel")

### ✅ Check Your Understanding: Advanced `while`

#### Exercise 7 — Write

Write a `while` loop that:
- Starts `port_number` at `8080`
- Checks ports from 8080 upward
- Breaks when it finds a port divisible by 7 (use `%`)
- Prints `"First port divisible by 7 above 8079: {port_number}"`

*Hint: 8085 is divisible by 7 (8085 / 7 = 1155)*

In [ ]:
# Exercise 7 — Find the first port above 8079 that is divisible by 7
# Your code here:


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
# 8080 % 7 = 1, 8081 % 7 = 2, ..., 8085 % 7 = 0 — first match

port_number = 8080

while True:
    if port_number % 7 == 0:
        break
    port_number += 1

print(f"First port divisible by 7 above 8079: {port_number}")
```

</details>

---
## 5. Nested Loop Control

### The problem: `break` only exits one level

When you have a nested loop, `break` only exits the **innermost** loop — the outer loop keeps running.

```python
for outer in range(3):
    for inner in range(3):
        if inner == 1:
            break  # only exits the inner loop
    # outer loop continues here
```

### Solution: the flag variable pattern

The standard Python approach is to use a **boolean flag** variable. The flag signals to the outer loop that it should also stop.

In [ ]:
# Nested loop control: stop scanning all IPs as soon as a critical finding is made
# Without a flag, breaking the inner loop would just move to the next IP

critical_port   = 445    # SMB — associated with ransomware propagation
critical_found  = False  # flag: becomes True when the critical port is found

for last_octet in range(1, 6):           # hosts .1 through .5
    for port_number in range(443, 448):  # ports 443–447
        print(f"Checking 192.168.1.{last_octet}:{port_number}")

        if port_number == critical_port:
            print(f"CRITICAL: SMB open on 192.168.1.{last_octet} — halting scan")
            critical_found = True
            break  # exits inner loop

    if critical_found:   # outer loop checks the flag each iteration
        break            # exits outer loop too

print("Scan complete")

### ✅ Check Your Understanding: Nested Loop Control

#### Exercise 8 — Fix

The code below is meant to stop scanning **all** ports and **all** hosts the moment port 22 is found. But it only stops the inner loop. Fix it using the flag pattern.

```python
for last_octet in range(1, 4):
    for port_number in range(20, 25):
        if port_number == 22:
            print(f"SSH found on .{last_octet} — stopping")
            break
        print(f"Scanning .{last_octet}:{port_number}")
```

In [ ]:
# Exercise 8 — Fix the nested loop so it stops completely when SSH is found
# Your corrected code here:


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
# Add a flag variable. Set it to True in the inner loop when SSH is found.
# Check the flag in the outer loop and break if it is True.

ssh_found = False

for last_octet in range(1, 4):
    for port_number in range(20, 25):
        if port_number == 22:
            print(f"SSH found on .{last_octet} — stopping entire scan")
            ssh_found = True
            break
        print(f"Scanning .{last_octet}:{port_number}")

    if ssh_found:
        break

print("Scan ended")
```

</details>

#### Exercise 9 — Write

Write a nested loop that:
- Outer loop: `range(1, 4)` (simulating 3 hosts)
- Inner loop: `range(1, 5)` (simulating 4 CVSS scores per host)
- Stops **all** scanning as soon as a score of 10 is found on any host
- Prints `"Critical score 10 found on host {h} — scan aborted"` when triggered
- Simulates: host 2, score 10 is the trigger

*Hint: use the flag pattern*

In [ ]:
# Exercise 9 — Abort a nested scan on critical finding using a flag
# Your code here:


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
# critical_found flag is set to True when score == 10 on host 2
# The outer loop checks the flag after each inner loop completes

critical_found = False

for host_id in range(1, 4):
    for cvss_score in range(1, 5):
        # Simulate: host 2 has a score of 10 on its 3rd check
        simulated_score = cvss_score * host_id  # host 2, score 3 → 6; host 2, score... let's do it simply
        actual_score = 10 if (host_id == 2 and cvss_score == 3) else cvss_score
        print(f"Host {host_id} — score: {actual_score}")

        if actual_score == 10:
            print(f"Critical score 10 found on host {host_id} — scan aborted")
            critical_found = True
            break

    if critical_found:
        break
```

</details>

---
## Summary

| Technique | Syntax | Use it when |
|---|---|---|
| `enumerate()` | `for idx, val in enumerate(seq, start=0):` | You need both the position and the value |
| `zip()` | `for a, b in zip(seq_a, seq_b):` | You want to iterate two sequences in parallel |
| `for…else` | `else:` after the loop body | You need to distinguish 'found via break' from 'exhausted loop' |
| Retry pattern | `while attempt < max: ... if success: break` | Repeating until success or giving up |
| Sentinel pattern | `while ...: if value == sentinel: break` | Processing a stream until a stop signal |
| Flag pattern | `found = False; if condition: found = True; break` | Exiting nested loops completely |

### Key rules to remember

- `enumerate()` default start is 0 — use `start=1` for human-readable numbering
- `zip()` stops at the shortest sequence — always make sure your sequences match in length when that matters
- `for…else` / `while…else`: the `else` runs only when **no** `break` occurred
- `break` exits only **one** level of nesting — use a flag to propagate exit upward

## Next Steps

Practise everything in **[02_Drills_Loops.ipynb](02_Drills_Loops.ipynb)**